<a href="https://colab.research.google.com/github/rg-smith/remote-sensing-hydro-2026/blob/main/lectures/lecture4-microwave.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 4: Microwave Remote Sensing
In this notebook, we will view SAR backscatter (amplitude) and passive microwave brightness. SAR phase data will be downloaded and viewed in a separate lab.

First we need to install a couple packages. If this shows an error after running, try the next code block. If it runs without an error, then you should be ok.

In [ ]:
!pip install geemap

In [2]:
import ee
import folium
import numpy as np
import branca.colormap as cm
import pandas as pd
import zipfile
import os
from tqdm import tqdm
import requests
import geemap

In [3]:
# you only need to run this once per session
ee.Authenticate()
ee.Initialize(project='replace with your code')

# Define custom functions for working with Google Earth Engine

In [13]:
# functions needed for this lab (and some other useful ones that you can use if you're interested)

# to convert a google earth engine image to a python array
def to_array(img,aoi):
  band_arrs = img.sampleRectangle(region=aoi,properties=['scale=1000'],defaultValue=-999)

  band_names=img.bandNames().getInfo()

  for kk in range(len(band_names)):
    if kk==0:
      dat1=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full=np.zeros((dat1.shape[0],dat1.shape[1],len(band_names)))
      dat_full[:,:,kk]=dat1
    else:
      dat=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full[:,:,kk]=dat
  return(dat_full)

# to calculate an index
def getIndex(image,b1,b2):
  return image.normalizedDifference([b1, b2])

# to calculate a ratio
def getRatio(image1,image2):
  ratio=image1.divide(image2)
  return ratio

# to create a color map from a specific image
def getVisparams(image,aoi):
  range = image.reduceRegion(ee.Reducer.percentile([1, 99]),aoi,300)
  vals = range.getInfo()
  min=list(vals.items())[0][1]
  max=list(vals.items())[1][1]
  visParams = {'min': min, 'max': max, "palette": ["red", "orange", "yellow", "cyan", "blue"]}
  return(visParams)

# to get the link to download an earth engine image
def getLink(image,aoi):
  link = image.getDownloadURL({
    'scale': 1000,
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': aoi})
  print(link)

# create an earth engine geometry polygon
def addGeometry(min_lon,max_lon,min_lat,max_lat):

  geom = ee.Geometry.Polygon(
      [[[min_lon, max_lat],
        [min_lon, min_lat],
        [max_lon, min_lat],
        [max_lon, max_lat]]])
  return(geom)

def get_center_from_geometry(geom_obj):
  centroid = geom_obj.centroid()
  coords = centroid.getInfo()['coordinates']
  return [coords[1], coords[0]] # Return as [latitude, longitude]

# to export an image to google drive
def export_to_drive(raster,filename,foldername,geometry):
  # Export the image, specifying scale and region.
  task = ee.batch.Export.image.toDrive(**{
      'image': raster,
      'description': filename,
      'folder': foldername,
      'fileNamePrefix': filename,
      'scale': 1000,
      'region': geometry,
      'fileFormat': 'GeoTIFF',
      'formatOptions': {
        'cloudOptimized': 'true'
      },
  })
  task.start()

def get_imgcollection(date1,date2,geometry,collection_name,band_name,function='mean'):
  collection = ee.ImageCollection(collection_name)
  collection = collection.filter(
    ee.Filter.listContains('system:band_names', band_name))
  if function=='mean':
      img = collection.filterDate(date1,date2).select(band_name).mean().clip(geometry)
  if function=='sum':
      img = collection.filterDate(date1,date2).select(band_name).sum().clip(geometry)
  return(img)

def get_img(geometry,collection_name,band_name):
  img = ee.Image(collection_name).select(band_name).clip(geometry)
  return(img)

# to get the link to download an earth engine image
def getLink(image,fname,aoi,scale=1000):
  link = image.getDownloadURL({
    'scale': scale,
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': aoi,
    'name': fname})
  # print(link)
  return(link)

def download_img(img,geom,fname,scale=1000):
    linkname = getLink(img,fname,geom,scale=scale)
    response = requests.get(linkname, stream=True)
    zipped = fname+'.zip'
    with open(zipped, "wb") as handle:
        for data in tqdm(response.iter_content()):
            handle.write(data)

    with zipfile.ZipFile(zipped, 'r') as zip_ref:
        zip_ref.extractall('')
    os.remove(zipped)

# Load and map SAR amplitude and microwave brightness

In [18]:
start='2021-06-01'
end='2021-09-30'

geom = addGeometry(-102, -95,37,40) # min long, max long, min lat, max lat (kansas)

center = get_center_from_geometry(geom)

In [19]:
# first create temporally reduced images

smap_v = get_imgcollection(start,end,geom,'NASA/SMAP/SPL3SMP_E/006','tb_v_corrected_am','mean')
vis_smap = getVisparams(smap_v,geom) # create visualization for ET
smap_layer_name = 'SMAP Brightness Temperature'

sar_v = get_imgcollection(start,end,geom,'COPERNICUS/S1_GRD','VV','mean')
vis_sar = getVisparams(sar_v,geom) # create visualization for ET
sar_layer_name = 'SAR Backscatter Amplitude'


In [ ]:
# generate map
Map = geemap.Map(center=center, zoom=6)
Map.add_basemap("HYBRID")

Map.addLayer(smap_v,vis_smap,smap_layer_name)
Map.add_colorbar(vis_smap,label=smap_layer_name,layer_name=smap_layer_name)

Map.addLayer(sar_v,vis_sar,sar_layer_name)
Map.add_colorbar(vis_sar,label=sar_layer_name,layer_name=sar_layer_name)

Map.addLayer(geom, {},"Study area")

Map.add_inspector()

Map #visualize the map